# ARC-3 serve-verify K3 — merge ladder checkpoint into Qwen3.6-27B and prove it serves

July's fine-tune died because a LoRA never reached the served model. This gate proves, in one
offline RTX-6000 commit, that a ladder checkpoint (default `checkpoint-16`, the OOD-early-peak arm):
1. **merges** into the base with exactly the trainer's load semantics (delta == B@A x alpha/r, checked),
2. **serves** under the duck's exact vLLM line (bf16; FP8 requant deferred to the publish step),
3. **behaves**: greedy val-prompt probes emit well-formed `<tool_call>` python calls,
4. differs from the BASE FP8 snapshot on the same probes (reported; precision-confounded).
Output = logs + `gate_result.json` only; the merged model stays in scratch by design (20GB cap).


In [ ]:
import glob, json, os, shutil, signal, subprocess, sys, time, urllib.request
deps = sorted(glob.glob("/kaggle/input/**/deps", recursive=True))
if deps: sys.path.insert(0, deps[0])
print("deps on path:", deps[:1])
import torch
DEV = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
RUN = "RTX PRO 6000" in DEV.upper() or os.environ.get("GATE_FORCE") == "1"
print(f"device: {DEV} | RUN={RUN}")
import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)
GATE_CKPT = os.environ.get("GATE_CKPT", "checkpoint-8")  # run-8 ladder: checkpoint-8 or sft_adapter (step-15 final)
GATE_MAX_TOKENS = int(os.environ.get("GATE_MAX_TOKENS", 6144))  # thinking is served-on; leave room before the tool_call


In [ ]:
CORPUS = os.path.dirname(sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True))[0])
sys.path.insert(0, CORPUS)
from sft_common import dequantize_fp8_inplace, strip_quantization_runtime
MODEL = next(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True)
             if "tokenizer_bundle" not in p and json.load(open(p)).get("model_type") == "qwen3_5")
# run-8 artifacts live in TWO shapes: sft_out/checkpoint-N (mid-run saves) and
# sft_adapter/ (the step-15 final, NOT under sft_out/) — glob must cover both.
_ckpt_candidates = sorted(glob.glob("/kaggle/input/**/sft_out/checkpoint-*", recursive=True))                  + sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/sft_adapter/adapter_config.json", recursive=True))
CKPT = next(p for p in _ckpt_candidates if p.rstrip("/").endswith(GATE_CKPT))
WHEELHOUSE = os.path.dirname(sorted(glob.glob("/kaggle/input/**/requirements.lock", recursive=True))[0])
print("model:", MODEL, "\nckpt:", CKPT, "\nwheelhouse:", WHEELHOUSE, "\ncorpus:", CORPUS)
TOOLS = json.load(open(os.path.join(CORPUS, "tools.json")))
val_rows = [json.loads(l) for l in open(os.path.join(CORPUS, "val.jsonl"))]
# fixed probes: first val row WITH an image, first val row WITHOUT, chosen deterministically
def has_image(r):
    return any(isinstance(m.get("content"), list) and any(p.get("type") == "image_url" for p in m["content"])
               for m in r["messages"])
PROBE_IMG = next(r for r in val_rows if has_image(r))
# corpus is fully multimodal (all 43 val rows carry images) -> probe B is just a distinct second row,
# text-only if one ever exists
PROBE_TXT = next((r for r in val_rows if not has_image(r)),
                 next(r for r in val_rows if r is not PROBE_IMG))
print("probe A (image):", len(PROBE_IMG["messages"]), "msgs | probe B:", len(PROBE_TXT["messages"]), "msgs")
MERGED = "/tmp/merged_" + GATE_CKPT


In [ ]:
if RUN:
    from transformers import AutoModelForImageTextToText, AutoProcessor
    t0 = time.time()
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL, torch_dtype=torch.bfloat16, device_map={"": 0})
    for a in ("quantization_config", "_pre_quantization_dtype"):
        if hasattr(model.config, a):
            try: setattr(model.config, a, None)
            except Exception: pass
    model.is_quantized = False
    if hasattr(model, "hf_quantizer"): model.hf_quantizer = None
    # 07-26 fix: apply FP8 scales BEFORE strip deletes them (v1 died in merge on f8 +=;
    # same root cause invalidated training runs 1-5)
    print("dequant:", dequantize_fp8_inplace(model))
    print("strip:", strip_quantization_runtime(model), f"| load {time.time()-t0:.0f}s")
    dt = {str(p.dtype) for p in model.parameters()}
    assert "torch.float8_e4m3fn" not in dt, dt

    # cache 3 targeted base weights for the post-merge delta assert
    acfg = json.load(open(os.path.join(CKPT, "adapter_config.json")))
    scaling = acfg["lora_alpha"] / acfg["r"]
    from peft import PeftModel
    pmodel = PeftModel.from_pretrained(model, CKPT, is_trainable=False)
    lora_mods = [(n, m) for n, m in pmodel.named_modules()
                 if hasattr(m, "lora_A") and "default" in getattr(m, "lora_A", {})]
    assert len(lora_mods) > 400, f"adapter did not attach: {len(lora_mods)} lora modules"
    import random; random.seed(0)
    checks = []
    for n, m in random.sample(lora_mods, 3):
        BA = (m.lora_B["default"].weight @ m.lora_A["default"].weight) * scaling
        checks.append((n, m.base_layer.weight.detach().clone(), BA.detach().clone()))
    # ---- NLL: adapter-attached vs base vs merged -------------------------------
    # The weight-level rel_err below is a proxy; THIS is the quantity that matters.
    # 2026-07-31: the merge assert failed with rel_err 0.64-0.75 across modules
    # spanning a 20x range of delta magnitudes. A synthetic reproduction showed
    # that is exactly what bf16 STORAGE of a delta ~1.5e-3 the size of the weights
    # produces (bf16 0.696 / fp16 0.138 / fp32 0.000) — and that accumulating the
    # merge in fp32 does NOT help, because the final cast is what destroys it.
    # So measure whether the merged model actually keeps the fine-tune's NLL gain,
    # rather than inferring model quality from weight fidelity.
    from sft_common import encode_with_mask
    _nll_rows = val_rows[:4]

    def _nll(mdl, tag):
        tot, ntok = 0.0, 0
        for r in _nll_rows:
            feats, _info = encode_with_mask(proc_for_nll, r["messages"], r["target"],
                                            TOOLS, 32768)
            if feats is None:
                continue
            feats = {k: (v.to(0) if hasattr(v, "to") else v) for k, v in feats.items()}
            with torch.no_grad():
                out = mdl(**feats)
            n = int((feats["labels"] != -100).sum())
            tot += float(out.loss) * n
            ntok += n
        v = tot / max(ntok, 1)
        print(f"[nll] {tag}: {v:.4f} over {ntok} target tokens", flush=True)
        return v

    from transformers import AutoProcessor as _AP
    try:
        proc_for_nll = _AP.from_pretrained(MODEL)
    except Exception:
        proc_for_nll = _AP.from_pretrained(os.path.join(CORPUS, "tokenizer_bundle"))

    nll_adapter = _nll(pmodel, "adapter attached (unmerged)")
    with pmodel.disable_adapter():
        nll_base = _nll(pmodel, "base (adapter disabled)")

    merged = pmodel.merge_and_unload()
    nll_merged = _nll(merged, "merged")

    gain_attached = nll_base - nll_adapter
    gain_merged = nll_base - nll_merged
    retained = gain_merged / gain_attached if abs(gain_attached) > 1e-6 else 0.0
    print(f"[nll] base={nll_base:.4f} attached={nll_adapter:.4f} merged={nll_merged:.4f}")
    print(f"[nll] gain attached={gain_attached:+.4f} merged={gain_merged:+.4f} "
          f"RETAINED={retained:.1%}", flush=True)
    results_nll = {"base": nll_base, "attached": nll_adapter, "merged": nll_merged,
                   "gain_attached": gain_attached, "gain_merged": gain_merged,
                   "retained_fraction": retained}
    # ----------------------------------------------------------------------------

    ok = []
    _named = dict(merged.named_modules())
    def _resolve(name):
        # peft's merge_and_unload strips the 'base_model.model.' prefix that
        # pre-merge names carry (KeyError on raw lookup, peft 0.19.x) —
        # audit fix 2026-07-26, mirrored in serving_assert._resolve_merged_module
        for c in (name, name.removeprefix("base_model.model."), "base_model.model." + name):
            if c in _named: return _named[c]
        raise KeyError(f"module {name!r} not found post-merge (tried prefix variants)")
    for (n, w_pre, BA) in checks:
        w_post = _resolve(n).weight.detach()
        delta = (w_post - w_pre).float(); exp = BA.float()
        rel = (delta - exp).norm() / (exp.norm() + 1e-9)
        ok.append({"module": n, "delta_norm": float(exp.norm()), "rel_err": float(rel)})
        print(f"[delta] {n}: |BA|={exp.norm():.4f} rel_err={rel:.4f}")
    # The weight-level delta is now DIAGNOSTIC, not pass/fail. A bf16 merge of a
    # delta this small is provably lossy (see the note above), so a high rel_err
    # here is expected and is not by itself evidence the adapter is broken — the
    # NLL block already measures what actually matters. Kept because delta_norm==0
    # would still mean the adapter never attached, which IS fatal.
    assert all(c["delta_norm"] > 0 for c in ok), f"adapter contributed nothing: {ok}"
    if any(c["rel_err"] >= 0.05 for c in ok):
        print(f"[warn] weight deltas degraded by merge rounding (expected for bf16): {ok}",
              flush=True)

    # A1-PROTOCOL §1 pre-registers: merged target-NLL gain >= 2% on the val rows.
    assert gain_attached > 0, (
        f"adapter gives NO NLL gain even attached — the checkpoint, not the merge, "
        f"is the problem: {results_nll}")
    rel_gain_merged = gain_merged / nll_base if nll_base else 0.0
    print(f"[nll] merged relative gain = {rel_gain_merged:.2%} (A1 §1 requires >= 2%)")
    assert rel_gain_merged >= 0.02, (
        f"MERGE GATE FAIL — merged model does not carry the fine-tune "
        f"({rel_gain_merged:.2%} < 2%; {retained:.1%} of the attached gain survived). "
        f"Serve the adapter unmerged via vLLM --enable-lora instead. {results_nll}")
    print("MERGE GATE: PASS")

    merged.config.torch_dtype = torch.bfloat16
    merged.config.use_cache = True
    t0 = time.time()
    merged.save_pretrained(MERGED, safe_serialization=True, max_shard_size="4GB")
    try:
        proc = AutoProcessor.from_pretrained(MODEL)
    except Exception as e:
        print("processor from snapshot failed:", e)
        proc = AutoProcessor.from_pretrained(os.path.join(CORPUS, "tokenizer_bundle"))
    proc.save_pretrained(MERGED)
    for extra in ("chat_template.jinja", "chat_template.json"):
        src = os.path.join(MODEL, extra)
        if os.path.exists(src) and not os.path.exists(os.path.join(MERGED, extra)):
            shutil.copy(src, MERGED)
    cfg = json.load(open(os.path.join(MERGED, "config.json")))
    for k in ("quantization_config", "_pre_quantization_dtype", "compression_config"):
        cfg.pop(k, None)
    json.dump(cfg, open(os.path.join(MERGED, "config.json"), "w"), indent=2)
    print(f"saved {MERGED} in {time.time()-t0:.0f}s:", sorted(os.listdir(MERGED))[:8], "...")
    del merged, pmodel, model, checks
    import gc; gc.collect(); torch.cuda.empty_cache()
    print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv"],
                         capture_output=True, text=True).stdout)


In [ ]:
if RUN:
    SITE = "/kaggle/working/vllm-site-packages"
    if not os.path.exists(os.path.join(SITE, "vllm")):
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                        "--find-links", WHEELHOUSE, "--requirement",
                        os.path.join(WHEELHOUSE, "requirements.lock"), "--target", SITE,
                        "--upgrade", "--ignore-installed", "--only-binary", ":all:",
                        "--no-compile", "--disable-pip-version-check", "--no-warn-conflicts"],
                       check=True, capture_output=True)
    ENV = dict(os.environ, PYTHONPATH=SITE, USE_TF="0", TRANSFORMERS_NO_TF="1",
               TRANSFORMERS_NO_TORCHVISION="1", VLLM_NO_USAGE_STATS="1")
    v = subprocess.run([sys.executable, "-c", "import vllm; print(vllm.__version__)"],
                       env=ENV, capture_output=True, text=True)
    print("vllm:", v.stdout.strip(), v.stderr.strip()[-200:])
    assert v.returncode == 0, "vLLM import failed from wheelhouse site-packages"


In [ ]:
BASE_URL = "http://127.0.0.1:1234/v1"

def req(url, payload=None, timeout=30):
    data = None if payload is None else json.dumps(payload).encode()
    r = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(r, timeout=timeout) as resp:
        return json.loads(resp.read().decode())

def start_server(model_path, log_path):
    # the duck's EXACT serve line (setup_commands.json), model path swapped
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
           "--model", model_path, "--served-model-name", "gate/model",
           "--host", "127.0.0.1", "--port", "1234", "--tensor-parallel-size", "1",
           "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder",
           "--generation-config", "vllm", "--enable-prefix-caching",
           "--default-chat-template-kwargs", '{"preserve_thinking": true}',
           "--reasoning-parser", "qwen3", "--max-model-len", "65536"]
    lh = open(log_path, "w")
    p = subprocess.Popen(cmd, env=ENV, stdout=lh, stderr=subprocess.STDOUT, text=True)
    deadline = time.monotonic() + 1800
    while time.monotonic() < deadline:
        if p.poll() is not None:
            print(open(log_path).read()[-4000:])
            raise RuntimeError(f"vLLM died rc={p.returncode} for {model_path}")
        try:
            req(BASE_URL + "/models", timeout=5); print("server ready:", model_path); return p
        except Exception:
            time.sleep(5)
    print(open(log_path).read()[-4000:])
    raise TimeoutError("vLLM never became ready")

def stop_server(p):
    p.send_signal(signal.SIGTERM)
    try: p.wait(60)
    except subprocess.TimeoutExpired: p.kill(); p.wait(30)
    for _ in range(36):
        used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True).stdout.strip()
        if used and int(used.split()[0]) < 8000: break
        time.sleep(5)
    print("gpu after stop:", used, "MiB")

def probe(tag):
    out = {}
    for name, row in (("val_img", PROBE_IMG), ("val_txt", PROBE_TXT)):
        r = req(BASE_URL + "/chat/completions",
                {"model": "gate/model", "messages": row["messages"], "tools": TOOLS,
                 "temperature": 0.0, "max_tokens": GATE_MAX_TOKENS}, timeout=900)
        msg = r["choices"][0]["message"]
        tcs = msg.get("tool_calls") or []
        wf = bool(tcs) and all(isinstance(json.loads(t["function"]["arguments"])
                                          if isinstance(t["function"]["arguments"], str)
                                          else t["function"]["arguments"], dict) for t in tcs)
        out[name] = {"tool_calls": [t["function"]["name"] for t in tcs], "well_formed": wf,
                     "content": (msg.get("content") or "")[:400],
                     "reasoning": (msg.get("reasoning_content") or "")[:200],
                     "raw_args": [str(t["function"]["arguments"])[:300] for t in tcs]}
        print(f"[{tag}:{name}] tools={out[name]['tool_calls']} well_formed={wf}")
    r = req(BASE_URL + "/chat/completions",
            {"model": "gate/model", "temperature": 0.0, "max_tokens": 128,
             "chat_template_kwargs": {"enable_thinking": False},
             "messages": [{"role": "user", "content": "Answer in one short sentence: what is 17 * 23?"}]},
            timeout=180)
    out["plain"] = {"content": r["choices"][0]["message"].get("content", "").strip()}
    print(f"[{tag}:plain] {out['plain']['content'][:120]}")
    return out


In [ ]:
if RUN:
    results = {"ckpt": GATE_CKPT}
    p = start_server(MERGED, "/kaggle/working/vllm-merged.log")
    results["merged"] = probe("merged")
    stop_server(p)
    p = start_server(MODEL, "/kaggle/working/vllm-base.log")
    results["base"] = probe("base")
    stop_server(p)

    differ = {k: results["merged"][k] != results["base"][k] for k in ("val_img", "val_txt", "plain")}
    results["outputs_differ"] = differ
    merged_wf = all(results["merged"][k]["well_formed"] for k in ("val_img", "val_txt"))
    results["verdict"] = {"merged_serves": True, "merged_tool_calls_well_formed": merged_wf,
                          "differs_from_base_anywhere": any(differ.values())}
    json.dump(results, open("/kaggle/working/gate_result.json", "w"), indent=2)
    print("\n" + "=" * 80)
    print(f"GATE VERDICT ({GATE_CKPT}): serves=True well_formed={merged_wf} differ={differ}")
    print("=" * 80)
    assert merged_wf, "merged model did not emit well-formed tool_calls on the val probes"
    print("SERVING-VERIFICATION GATE: PASS")
